# 练习实验：使用 Deep Learning 实现 Content-based Filtering

<img align="left" src="./images/film_strip_vertical.png" style="width:40px;" alt="胶片">

本练习将使用 Neural Network 实现 Content-based Filtering（基于内容的过滤），构建电影推荐系统。

## 目录

- [1 - 软件包](#1)
- [2 - 电影评分数据集](#2)
- [3 - 使用 Neural Network 进行 Content-based Filtering](#3)
  - [3.1 训练数据](#3.1)
  - [3.2 准备训练数据](#3.2)
- [4 - 构建 Content-based Filtering 网络](#4)
  - [练习 1](#ex01)
- [5 - 预测](#5)
  - [5.1 为新用户预测](#5.1)
  - [5.2 为已有用户预测](#5.2)
  - [5.3 查找相似物品](#5.3)
    - [练习 2](#ex02)
- [6 - 恭喜完成](#6)


> **注意：** 为避免 autograder（自动评分器）报错，提交作业时请勿编辑或删除不参与评分的单元，也不要新增单元。
>
> **通过作业后**，如果希望尝试修改不参与评分的代码，可以按照本 Notebook 底部的说明操作。


<a name="1"></a>
## 1 - 软件包

<img align="left" src="./images/movie_camera.png" style="width:40px;" alt="电影摄影机">

使用已熟悉的 NumPy、TensorFlow 和 [Scikit-learn](https://scikit-learn.org/stable/) 辅助函数。此外，用 [tabulate](https://pypi.org/project/tabulate/) 整齐打印表格，用 [Pandas](https://pandas.pydata.org/) 组织表格数据。


In [ ]:
import numpy as np
import numpy.ma as ma
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import tabulate
from recsysNN_utils import *
pd.set_option("display.precision", 1)

<a name="2"></a>
## 2 - 电影评分数据集

<img align="left" src="./images/film_rating.png" style="width:40px;" alt="电影评分">

数据来自 [MovieLens 的 ml-latest-small 数据集](https://grouplens.org/datasets/movielens/latest/)。相关文献：F. Maxwell Harper and Joseph A. Konstan, 2015, *The MovieLens Datasets: History and Context*, ACM Transactions on Interactive Intelligent Systems (TiiS), 5(4), 19:1–19:19，[DOI](https://doi.org/10.1145/2827872)。

课程使用的原始数据约包含 600 名用户对 9,000 部电影的评分，范围为 0.5–5，步长为 0.5。本实验筛选了 2000 年以来的电影与热门 Genre，得到 $n_u=397$ 名用户、$n_m=847$ 部电影和 25,521 条评分。

每部电影有片名、上映年份和一个或多个 Genre。例如，*Toy Story 3* 于 2010 年上映，类别为 `Adventure|Animation|Children|Comedy|Fantasy`。除评分外，数据集几乎没有其他用户信息。下面将据此构建 Neural Network 的训练向量。

下表展示按评分人数排序的前十部电影，它们的平均评分也较高。你看过其中多少部？


In [ ]:
top10_df = pd.read_csv("./data/content_top10_df.csv")
bygenre_df = pd.read_csv("./data/content_bygenre_df.csv")
top10_df

下表按 Genre 汇总信息。不同 Genre 的评分数量差异很大。由于一部电影可能属于多个 Genre，表中评分数量的总和会超过原始评分总数。


In [ ]:
bygenre_df

<a name="3"></a>
## 3 - 使用 Neural Network 进行 Content-based Filtering

在 Collaborative Filtering 实验中，用户向量与电影向量仅根据评分学习得到，两者的 Dot Product 用于预测评分。

Content-based Filtering 同样生成用户与电影的 Feature Vectors，但还可以利用用户、电影的其他信息来改善预测。这些信息先输入 Neural Networks，再生成向量，如下图所示：

<figure style="text-align:center;">
<img src="./images/RecSysNN.png" style="width:500px;height:280px;" alt="分别编码用户与电影信息的双网络结构">
</figure>

<a name="3.1"></a>
### 3.1 训练数据

电影输入包含原始特征与 Engineered Features。可回顾 Course 1 Week 2 的 Feature Engineering 实验：原始特征是上映年份和 14 个 Genre 的指示向量，一部电影可同时属于多个 Genre；构造的特征是根据用户评分得到的电影平均分。

用户输入由 Engineered Features 构成：计算每个用户对各 Genre 的平均评分。数据还携带 User ID、评分数量和总体平均分，方便理解结果，但它们不作为训练或预测时的输入特征。

训练样本来自数据集中的评分。为增加较少出现的 Genre 的样本数量，部分评分会重复使用。数据保存为两个行数相同的数组：用户数组与电影数组，每行对应同一条评分。

下面加载并查看部分数据。


In [ ]:
# Load Data, set configuration variables
item_train, user_train, y_train, item_features, user_features, item_vecs, movie_dict, user_to_genre = load_data()

num_user_features = user_train.shape[1] - 3  # remove userid, rating count and ave rating during training
num_item_features = item_train.shape[1] - 1  # remove movie id at train time
uvs = 3  # user genre vector start
ivs = 3  # item genre vector start
u_s = 3  # start of columns to use in training, user
i_s = 1  # start of columns to use in training, items
print(f"Number of training vectors: {len(item_train)}")

先查看用户训练数组的前几行。


In [ ]:
pprint_train(user_train, user_features, uvs,  u_s, maxcount=5)

部分用户与电影字段不参与训练。上表中用方括号 `[]` 标记的 `user id`、`rating count`、`rating ave` 等字段，不作为模型训练或预测的输入。

上面显示了用户 2 对各 Genre 的平均评分，0 表示该用户未评价过这一类电影。同一个用户对不同电影评分时，使用的用户输入向量相同。下面查看电影数组的前几行。


In [ ]:
pprint_train(item_train, item_features, ivs, i_s, maxcount=5, user=False)

电影数组包含上映年份、平均评分，以及各 Genre 的指示值。电影属于某个 Genre 时，对应值为 1。Movie ID 不用于训练，但便于解释数据。


In [ ]:
print(f"y_train[:5]: {y_train[:5]}")

目标 `y` 是用户给这部电影的评分。


例如，电影 6874 于 2003 年上映，属于 Action、Crime 和 Thriller。用户 2 对 Action 电影的平均评分为 3.9，该电影在 MovieLens 中的平均分为 4。这里 `y=4`，表示用户 2 对电影 6874 也给出了 4 分。

一个训练样本由用户数组的一行、电影数组的对应行，以及 `y_train` 中的对应评分组成。


<a name="3.2"></a>
### 3.2 准备训练数据

Course 1 Week 2 介绍了使用 Feature Scaling 改善收敛。这里用 Scikit-learn 的 [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) 缩放输入特征，并演示用 `inverse_transform` 恢复原始输入。

目标评分则通过 [`MinMaxScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html) 缩放到 -1 到 1 之间。


In [ ]:
# scale training data
item_train_unscaled = item_train
user_train_unscaled = user_train
y_train_unscaled    = y_train

scalerItem = StandardScaler()
scalerItem.fit(item_train)
item_train = scalerItem.transform(item_train)

scalerUser = StandardScaler()
scalerUser.fit(user_train)
user_train = scalerUser.transform(user_train)

scalerTarget = MinMaxScaler((-1, 1))
scalerTarget.fit(y_train.reshape(-1, 1))
y_train = scalerTarget.transform(y_train.reshape(-1, 1))
#ynorm_test = scalerTarget.transform(y_test.reshape(-1, 1))

print(np.allclose(item_train_unscaled, scalerItem.inverse_transform(item_train)))
print(np.allclose(user_train_unscaled, scalerUser.inverse_transform(user_train)))

为了评估模型，按照 Course 2 Week 3 的方法划分 Training Set 和 Test Set。这里使用 Scikit-learn 的 [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) 打乱并划分数据。分别处理电影、用户和 `y` 时，使用相同的 `random_state`，确保它们以相同顺序打乱，保持样本对应关系。


In [ ]:
item_train, item_test = train_test_split(item_train, train_size=0.80, shuffle=True, random_state=1)
user_train, user_test = train_test_split(user_train, train_size=0.80, shuffle=True, random_state=1)
y_train, y_test       = train_test_split(y_train,    train_size=0.80, shuffle=True, random_state=1)
print(f"movie/item training data shape: {item_train.shape}")
print(f"movie/item test data shape: {item_test.shape}")

经过 StandardScaler 缩放的输入特征整体均值为零；打乱后的子集均值通常接近零。


In [ ]:
pprint_train(user_train, user_features, uvs, u_s, maxcount=5)

<a name="4"></a>
## 4 - 构建 Content-based Filtering 网络

按照前图构建两个网络，再通过 Dot Product 合并输出。本例两个网络采用相同架构，但这并非必要条件。如果用户信息明显比电影信息丰富，可以采用更复杂的用户网络。本例两类信息规模接近，因此使用相同结构。

<a name="ex01"></a>
### 练习 1

分别用 Keras Sequential Model 构建：

- 第 1 层：256 个单元的 Dense Layer，使用 ReLU。
- 第 2 层：128 个单元的 Dense Layer，使用 ReLU。
- 第 3 层：`num_outputs` 个单元的 Dense Layer，使用 Linear Activation，或不指定 Activation。

网络的其余连接代码已提供。它使用 Keras [Functional API](https://keras.io/guides/functional_api/)，以便更灵活地连接各组件。


In [ ]:
# GRADED_CELL
# UNQ_C1

num_outputs = 32
tf.random.set_seed(1)
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  
  
  
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  
  
  
    ### END CODE HERE ###  
])

# create the user input and point to the base network
input_user = tf.keras.layers.Input(shape=(num_user_features))
vu = user_NN(input_user)
vu = tf.linalg.l2_normalize(vu, axis=1)

# create the item input and point to the base network
input_item = tf.keras.layers.Input(shape=(num_item_features))
vm = item_NN(input_item)
vm = tf.linalg.l2_normalize(vm, axis=1)

# compute the dot product of the two vectors vu and vm
output = tf.keras.layers.Dot(axes=1)([vu, vm])

# specify the inputs and output of the model
model = tf.keras.Model([input_user, input_item], output)

model.summary()

In [ ]:
# Public tests
from public_tests import *
test_tower(user_NN)
test_tower(item_NN)

<details>
<summary><strong>点击查看提示</strong></summary>

可以如下创建使用 ReLU 的 Dense Layer：

```python
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),


    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),


    ### END CODE HERE ###  
])
```

<details>
<summary><strong>点击查看参考实现</strong></summary>

```python
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])
```

</details>
</details>


使用 Mean Squared Error Loss 和 Adam Optimizer。


In [ ]:
tf.random.set_seed(1)
cost_fn = tf.keras.losses.MeanSquaredError()
opt = keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=opt,
              loss=cost_fn)

In [ ]:
tf.random.set_seed(1)
model.fit([user_train[:, u_s:], item_train[:, i_s:]], y_train, epochs=30)

评估模型，计算 Test Set 上的 Loss。


In [ ]:
model.evaluate([user_test[:, u_s:], item_test[:, i_s:]], y_test)

它与 Training Loss 接近，说明模型没有表现出明显的 Overfitting。


<a name="5"></a>
## 5 - 预测

下面在几种不同场景下使用模型进行预测。

<a name="5.1"></a>
### 5.1 - 为新用户预测

先创建一个新用户，并为其推荐电影。试过示例偏好后，可以改成自己的偏好，观察推荐变化。评分范围为 0.5–5.0，步长为 0.5。


In [ ]:
new_user_id = 5000
new_rating_ave = 0.0
new_action = 0.0
new_adventure = 5.0
new_animation = 0.0
new_childrens = 0.0
new_comedy = 0.0
new_crime = 0.0
new_documentary = 0.0
new_drama = 0.0
new_fantasy = 5.0
new_horror = 0.0
new_mystery = 0.0
new_romance = 0.0
new_scifi = 0.0
new_thriller = 0.0
new_rating_count = 3

user_vec = np.array([[new_user_id, new_rating_count, new_rating_ave,
                      new_action, new_adventure, new_animation, new_childrens,
                      new_comedy, new_crime, new_documentary,
                      new_drama, new_fantasy, new_horror, new_mystery,
                      new_romance, new_scifi, new_thriller]])

这个新用户喜欢 Adventure 和 Fantasy。下面寻找模型预测评分最高的电影。

`item_vecs` 为 Training Set 与 Test Set 中的每部电影保存一个输入向量。将它们与新用户向量配对，缩放后输入模型，即可预测该用户对所有电影的评分。


In [ ]:
# generate and replicate the user vector to match the number movies in the data set.
user_vecs = gen_user_vecs(user_vec,len(item_vecs))

# scale our user and item vectors
suser_vecs = scalerUser.transform(user_vecs)
sitem_vecs = scalerItem.transform(item_vecs)

# make a prediction
y_p = model.predict([suser_vecs[:, u_s:], sitem_vecs[:, i_s:]])

# unscale y prediction 
y_pu = scalerTarget.inverse_transform(y_p)

# sort the results, highest prediction first
sorted_index = np.argsort(-y_pu,axis=0).reshape(-1).tolist()  #negate to get largest rating first
sorted_ypu   = y_pu[sorted_index]
sorted_items = item_vecs[sorted_index]  #using unscaled vectors for display

print_pred_movies(sorted_ypu, sorted_items, movie_dict, maxcount = 10)

<a name="5.2"></a>
### 5.2 - 为已有用户预测

以数据集中的用户 2 为例，比较模型预测评分与该用户的实际评分。


In [ ]:
uid = 2 
# form a set of user vectors. This is the same vector, transformed and repeated.
user_vecs, y_vecs = get_user_vecs(uid, user_train_unscaled, item_vecs, user_to_genre)

# scale our user and item vectors
suser_vecs = scalerUser.transform(user_vecs)
sitem_vecs = scalerItem.transform(item_vecs)

# make a prediction
y_p = model.predict([suser_vecs[:, u_s:], sitem_vecs[:, i_s:]])

# unscale y prediction 
y_pu = scalerTarget.inverse_transform(y_p)

# sort the results, highest prediction first
sorted_index = np.argsort(-y_pu,axis=0).reshape(-1).tolist()  #negate to get largest rating first
sorted_ypu   = y_pu[sorted_index]
sorted_items = item_vecs[sorted_index]  #using unscaled vectors for display
sorted_user  = user_vecs[sorted_index]
sorted_y     = y_vecs[sorted_index]

#print sorted predictions for movies rated by the user
print_existing_user(sorted_ypu, sorted_y.reshape(-1,1), sorted_user, sorted_items, ivs, uvs, movie_dict, maxcount = 50)

模型预测通常与实际评分相差不超过 1 分，但还无法非常准确地预测用户对每一部具体电影的评价，尤其是在该评分与用户对该 Genre 的平均评分差异很大时。可以更换上方的 User ID，查看其他用户。并非所有 User ID 都出现在筛选后的 Training Set 中。


<a name="5.3"></a>
### 5.3 - 查找相似物品

上述网络生成用户 Feature Vector $\mathbf{v}_u$ 和电影 Feature Vector $\mathbf{v}_m$，本例均为 32 维。虽然各维度不易直接解释，但相似电影通常具有相近的向量，因此可用于推荐。例如，用户给 *Toy Story 3* 高分时，可以推荐电影向量与其接近的其他电影。

可使用两个向量的 Squared Distance 衡量差异，距离越小越相似：

$$\left\|\mathbf{v}_m^{(k)}-\mathbf{v}_m^{(i)}\right\|_2^2=\sum_{l=1}^{n}\left(v_{m,l}^{(k)}-v_{m,l}^{(i)}\right)^2.\tag{1}$$


<a name="ex02"></a>
### 练习 2

编写计算 Squared Distance 的函数。


In [ ]:
# GRADED_FUNCTION: sq_dist
# UNQ_C2
def sq_dist(a,b):
    """
    Returns the squared distance between two vectors
    Args:
      a (ndarray (n,)): vector with n features
      b (ndarray (n,)): vector with n features
    Returns:
      d (float) : distance
    """
    ### START CODE HERE ###     
    
    ### END CODE HERE ###     
    return d

In [ ]:
a1 = np.array([1.0, 2.0, 3.0]); b1 = np.array([1.0, 2.0, 3.0])
a2 = np.array([1.1, 2.1, 3.1]); b2 = np.array([1.0, 2.0, 3.0])
a3 = np.array([0, 1, 0]);       b3 = np.array([1, 0, 0])
print(f"squared distance between a1 and b1: {sq_dist(a1, b1):0.3f}")
print(f"squared distance between a2 and b2: {sq_dist(a2, b2):0.3f}")
print(f"squared distance between a3 and b3: {sq_dist(a3, b3):0.3f}")

**预期输出**

```text
squared distance between a1 and b1: 0.000
squared distance between a2 and b2: 0.030
squared distance between a3 and b3: 2.000
```


In [ ]:
# Public tests
test_sq_dist(sq_dist)

<details>
<summary><strong>点击查看提示</strong></summary>

虽然公式中有求和，这里仍可直接用 Vectorization：先逐元素相减，用 `np.square` 将结果逐元素平方，最后用 `np.sum` 求和。

</details>


模型训练完成后，可以一次性计算电影之间的距离矩阵，后续生成推荐时复用，无需重新训练。首先，用已训练的 `item_NN` 构建一个小模型，将每部电影的输入向量转换为 Feature Vector $\mathbf{v}_m$。


In [ ]:
input_item_m = tf.keras.layers.Input(shape=(num_item_features))    # input layer
vm_m = item_NN(input_item_m)                                       # use the trained item_NN
vm_m = tf.linalg.l2_normalize(vm_m, axis=1)                        # incorporate normalization as was done in the original model
model_m = tf.keras.Model(input_item_m, vm_m)                                
model_m.summary()

构建电影模型后，将所有电影输入向量 `item_vecs` 按训练时相同的方式缩放，再输入模型。结果是每部电影对应的 32 维 Feature Vector。


In [ ]:
scaled_item_vecs = scalerItem.transform(item_vecs)
vms = model_m.predict(scaled_item_vecs[:,i_s:])
print(f"size of all predicted movie feature vectors: {vms.shape}")

下面计算各电影 Feature Vector 与所有其他电影 Feature Vectors 之间的 Squared Distance，组成距离矩阵：

<figure>
<img src="./images/distmatrix.PNG" style="width:400px;height:225px;" alt="电影之间的 Squared Distance 矩阵">
</figure>


每一行的最小值对应最相似的电影。为避免选中电影自身，使用 NumPy 的 [Masked Arrays](https://numpy.org/doc/1.21/user/tutorial-ma.html) 屏蔽对角线，这些值不参与最小值计算。


In [ ]:
count = 50  # number of movies to display
dim = len(vms)
dist = np.zeros((dim,dim))

for i in range(dim):
    for j in range(dim):
        dist[i,j] = sq_dist(vms[i, :], vms[j, :])
        
m_dist = ma.masked_array(dist, mask=np.identity(dist.shape[0]))  # mask the diagonal

disp = [["movie1", "genres", "movie2", "genres"]]
for i in range(count):
    min_idx = np.argmin(m_dist[i])
    movie1_id = int(item_vecs[i,0])
    movie2_id = int(item_vecs[min_idx,0])
    disp.append( [movie_dict[movie1_id]['title'], movie_dict[movie1_id]['genres'],
                  movie_dict[movie2_id]['title'], movie_dict[movie1_id]['genres']]
               )
table = tabulate.tabulate(disp, tablefmt='html', headers="firstrow")
table

结果显示，模型通常会推荐 Genre 相似的电影。


<a name="6"></a>
## 6 - 恭喜完成！

<img align="left" src="./images/film_award.png" style="width:40px;" alt="电影奖杯">

你已经完成了一个 Content-based Recommender System。这种结构是许多商业推荐系统的基础。若有更多用户信息，可以进一步扩充用户输入；推荐对象也不限于电影，还可以是书籍、汽车，或与购物车中商品相似的其他商品。


<details>
<summary><span style="color:darkgreen;"><strong>如需尝试修改不参与评分的代码，请点击这里。</strong></span></summary>

**重要提示：请在通过作业后再执行这些操作，以免影响 autograder。**

1. 在 Notebook 菜单中，依次选择 `View` > `Cell Toolbar` > `Edit Metadata`。
2. 在希望锁定或解锁的代码单元旁，点击 `Edit Metadata`。
3. 将 `editable` 属性设为 `true` 以解锁，或设为 `false` 以锁定。
4. 在 Notebook 菜单中，依次选择 `View` > `Cell Toolbar` > `None`。

下面的简短演示展示了上述步骤：

<img src="https://lh3.google.com/u/0/d/14Xy_Mb17CZVgzVAgq7NCjMVBvSae3xO1" alt="解锁 Notebook 单元的操作演示（unlock_cells.gif）">

</details>
